# ACME: BEV pedestrian trajectories

Overhead (bird's-eye view) cameras recorded the scenes around the robots. Every
pedestrian was tracked in these videos and verified by annotators. Per session the
dataset contains:

| file | content |
|---|---|
| `trajectory-transform/trajectories/<dataset>/<session>.toml` | pixel tracks at 10 FPS (`human<N>`: `traj_start`, `trajectories`) |
| `trajectory-transform/homographies/<dataset>/<session>_homography.txt` | 3x3 pixel → metre homography |
| `frames/<stem>.jpg`, `traj_frames/<stem>.jpg` | BEV frame, and the same frame with tracks drawn in |
| `annotations/<stem>.json`, `masks/<stem>/tag_<id>_<name>.jpg` | semantic regions (open area, corridors, ...) |

This notebook walks through loading, converting to metres, cleaning and analysing
the tracks with `acme_dataset.bev` (a library version of `explore_bev.ipynb`).

In [ ]:
from pathlib import Path
import os

# Root of the dataset: the folder that contains `ros2 bags/`, `human_readable_sampled/`,
# `bev/` and `dataset_tags.xlsx`. Override with the ACME_ROOT environment variable.
DATA_ROOT = Path(os.environ.get("ACME_ROOT", "../ACME-dataset"))
BEV_ROOT = Path(os.environ.get("ACME_BEV_ROOT", DATA_ROOT / "bev"))

DOWNLOAD = False
if DOWNLOAD:
    from acme_dataset import download
    download("bev", local_dir=DATA_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from acme_dataset import BEVIndex, BEVSession

index = BEVIndex(BEV_ROOT)          # one walk over the annotation tree
sessions = index.sessions()
display(sessions.groupby("dataset").size().rename("sessions").to_frame())
sessions.head()

## 1. Pick a session

In [ ]:
DATASET, SESSION_ID = "bonn", "20"
if (DATASET, SESSION_ID) not in index.trajectories:
    DATASET, SESSION_ID = sessions.iloc[0][["dataset", "session"]]
session = BEVSession(index, DATASET, SESSION_ID)
print(session, "| stem:", session.stem, "| homography:", session.homography_path.name,
      "| lens model:", session.intrinsics is not None)

## 2. Frame, semantic regions and raw tracks

In [ ]:
from acme_dataset.viz import plot_bev_session

raw = session.trajectories(filtered=False)
print(f"{raw.ped_id.nunique()} annotated pedestrians, {len(raw)} points at 10 FPS")
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plot_bev_session(session, raw, ax=axes[0])
if session.traj_frame() is not None:
    axes[1].imshow(session.traj_frame())
    axes[1].set_title("annotation reference")
axes[1].axis("off");

## 3. Metric, cleaned trajectories

`trajectories()` maps pixels to metres with the homography (undistorting first when
the BEV camera has a lens model, as for CMU) and applies the standard filter:
drop tracks shorter than 3.2 s, interpolate annotation jumps, smooth (5 frames) and
label points as `dynamic`, `stationary` or `out_of_bounds`.

* `target_fps=2.5` reproduces the downsampled output of `transform_trajectories.py`.
* `coordinate_bounds=(-25, 25)` marks points far from the homography origin as `out_of_bounds`.

In [ ]:
clean = session.trajectories(coordinate_bounds=(-25, 25))
print(f"{raw.ped_id.nunique()} -> {clean.ped_id.nunique()} pedestrians after filtering")
display(clean.head())
display(clean.label.value_counts(normalize=True).round(3).to_frame("fraction"))

colors = {"dynamic": "#00a65a", "stationary": "#ff9800", "out_of_bounds": "#e53935"}
fig, ax = plt.subplots(figsize=(8, 8))
for label, part in clean.groupby("label"):
    ax.scatter(part.x, part.y, s=1, c=colors[label], label=label)
ax.set(xlabel="x (m)", ylabel="y (m)", aspect="equal", title="Filtered metric trajectories")
ax.legend(markerscale=8);

## 4. Split by semantic region

Regions can overlap, so one stretch of a track can belong to several tags.

In [ ]:
segments = session.tag_segments(clean)
if segments.empty:
    print("this session has no tag masks")
else:
    display(segments.groupby("tag_name").agg(segments=("segment", "nunique"), pedestrians=("ped_id", "nunique")))

## 5. Walking speed per region

Instantaneous speed between consecutive dynamic points (capped at 5 m/s).

In [ ]:
def step_speeds(df, fps=10.0):
    df = df.sort_values(["segment", "frame"]) if "segment" in df else df.sort_values(["ped_id", "frame"])
    key = df["segment"] if "segment" in df else df["ped_id"]
    same = (key.values[1:] == key.values[:-1]) & (np.diff(df["frame"].values) == 1)
    dyn = (df["label"].values[1:] == "dynamic") & (df["label"].values[:-1] == "dynamic")
    speed = np.hypot(np.diff(df["x"].values), np.diff(df["y"].values)) * fps
    keep = same & dyn & (speed < 5.0)
    return pd.DataFrame({"speed": speed[keep], **({"tag_name": df["tag_name"].values[1:][keep]} if "tag_name" in df else {})})

speeds = step_speeds(segments if not segments.empty else clean)
if "tag_name" in speeds:
    display(speeds.groupby("tag_name")["speed"].describe().round(2))
speeds["speed"].plot.hist(bins=40, figsize=(7, 3), title="walking speed (m/s)");

## 6. Closest approach between pedestrians

For every pair of pedestrians visible at the same time, the minimum distance over their shared frames.

In [ ]:
dyn = clean[clean.label == "dynamic"]
by_frame = dyn.pivot_table(index="frame", columns="ped_id", values=["x", "y"])
peds = by_frame["x"].columns
xs, ys = by_frame["x"].to_numpy(), by_frame["y"].to_numpy()
pairs = []
for a in range(len(peds)):
    dist = np.hypot(xs[:, a:a + 1] - xs[:, a + 1:], ys[:, a:a + 1] - ys[:, a + 1:])
    for k, d in enumerate(np.nanmin(np.where(np.isnan(dist), np.inf, dist), axis=0)):
        if np.isfinite(d):
            pairs.append((peds[a], peds[a + 1 + k], d))
pairs = pd.DataFrame(pairs, columns=["ped_a", "ped_b", "min_distance_m"])
close = pairs[pairs.min_distance_m < 2.0]
print(f"{len(pairs)} co-present pairs, {len(close)} closer than 2 m, {int((close.min_distance_m < 1).sum())} closer than 1 m")
close.min_distance_m.plot.hist(bins=30, figsize=(7, 3), title="closest approach < 2 m");

## 7. Export

All sessions can be written to CSV (metres, one file per session) from the shell:

```bash
acme-dataset bev-export ACME-dataset/bev --out bev_csv --target-fps 2.5   # trajectory-prediction style
acme-dataset bev-export ACME-dataset/bev --out bev_csv_tags --tags            # with semantic regions
```